# Unidad 6 — Redes neuronales para visión
## Análisis de MobileNetV3-Small para comandos visuales en Webots

El entrenamiento se ejecuta mediante `train.py`. El cuaderno consume los artefactos guardados para analizar convergencia, desempeño por clase, errores y respuesta ante imágenes de un ambiente independiente.

### Flujo experimental

```text
dataset de Webots → entrenamiento reproducible → mejor checkpoint
                   → métricas y predicciones → análisis
```

Ejecución recomendada desde la carpeta del proyecto:

```powershell
python train.py --epochs 15 --head-epochs 3
```

No se utiliza inversión horizontal como aumento: transformaría una señal izquierda en derecha sin modificar su etiqueta.

In [1]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

project_root = Path.cwd().resolve()
if not (project_root / 'train.py').exists():
    candidates = [path for path in project_root.parents if (path / 'train.py').exists()]
    if not candidates:
        raise FileNotFoundError('No se encontró train.py. Ejecute el cuaderno desde la carpeta del proyecto.')
    project_root = candidates[0]

run_dir = Path(os.environ.get('VISION_RUN_DIR', project_root / 'runs' / 'mobilenet_v3_small')).resolve()
required = ['config.json', 'history.csv', 'metrics.json', 'confusion_matrix.csv',
            'test_predictions.csv', 'best_model.pth']
missing = [name for name in required if not (run_dir / name).exists()]
if missing:
    raise FileNotFoundError(f'Faltan artefactos en {run_dir}: {missing}. Ejecute train.py primero.')

print(f'Proyecto: {project_root}')
print(f'Resultados: {run_dir}')

FileNotFoundError: Faltan artefactos en C:\Users\20808\Documents\Repositorios\Vision_en_Robotica\deep_learning\01_comandos_visuales_webots\runs\mobilenet_v3_small: ['config.json', 'history.csv', 'metrics.json', 'confusion_matrix.csv', 'test_predictions.csv', 'best_model.pth']. Ejecute train.py primero.

In [ ]:
config = json.loads((run_dir / 'config.json').read_text(encoding='utf-8'))
metrics = json.loads((run_dir / 'metrics.json').read_text(encoding='utf-8'))
history = pd.read_csv(run_dir / 'history.csv')
predictions = pd.read_csv(run_dir / 'test_predictions.csv')
confusion = pd.read_csv(run_dir / 'confusion_matrix.csv', index_col=0)
classes = config['classes']

summary = pd.Series({
    'dispositivo': config['device'],
    'pesos iniciales': config['weights'],
    'mejor época': metrics['best_epoch'],
    'exactitud de validación': metrics['best_val_accuracy'],
    'exactitud de prueba': metrics['test_accuracy'],
    'muestras de prueba': metrics['test_samples'],
    'tiempo de entrenamiento (s)': metrics['training_seconds'],
}, name='valor')
summary.to_frame()

### Convergencia

La pérdida mide el desacuerdo entre las probabilidades predichas y la clase real. La exactitud representa la fracción de imágenes clasificadas correctamente. La diferencia entre entrenamiento y validación ayuda a identificar sobreajuste.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history['epoch'], history['train_loss'], marker='o', label='entrenamiento')
axes[0].plot(history['epoch'], history['val_loss'], marker='o', label='validación')
axes[0].set(xlabel='Época', ylabel='Pérdida', title='Evolución de la pérdida')
axes[1].plot(history['epoch'], history['train_accuracy'], marker='o', label='entrenamiento')
axes[1].plot(history['epoch'], history['val_accuracy'], marker='o', label='validación')
axes[1].set(xlabel='Época', ylabel='Exactitud', title='Evolución de la exactitud', ylim=(0, 1.05))
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()
plt.show()

### Matriz de confusión y métricas por clase

Las filas corresponden a la clase real y las columnas a la predicción. Los valores fuera de la diagonal muestran qué órdenes podrían confundirse durante la operación del robot.

In [ ]:
matrix = confusion.to_numpy()
fig, axis = plt.subplots(figsize=(6, 5))
image = axis.imshow(matrix, cmap='Greys')
axis.set_xticks(range(len(classes)), classes, rotation=35, ha='right')
axis.set_yticks(range(len(classes)), classes)
axis.set(xlabel='Predicción', ylabel='Clase real', title='Matriz de confusión — prueba')
threshold = matrix.max() / 2 if matrix.size else 0
for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        color = 'white' if matrix[row, column] > threshold else 'black'
        axis.text(column, row, matrix[row, column], ha='center', va='center', color=color)
fig.colorbar(image, ax=axis, label='Imágenes')
plt.tight_layout()
plt.show()

report = pd.DataFrame(metrics['classification_report']).T
report.loc[classes, ['precision', 'recall', 'f1-score', 'support']]

### Inspección de predicciones

La confianza es la probabilidad máxima producida por la red. Una predicción correcta con confianza baja puede indicar una condición cercana al límite de decisión. Una predicción incorrecta con confianza alta requiere atención porque podría producir una orden robótica equivocada.

In [ ]:
if predictions['correct'].dtype != bool:
    predictions['correct'] = predictions['correct'].astype(str).str.lower().eq('true')

fig, axis = plt.subplots(figsize=(7, 4))
for correct, label in [(True, 'correctas'), (False, 'incorrectas')]:
    values = predictions.loc[predictions['correct'] == correct, 'confidence']
    if not values.empty:
        axis.hist(values, bins=np.linspace(0, 1, 11), alpha=0.65, label=label)
axis.set(xlabel='Confianza', ylabel='Número de imágenes', title='Confianza de las predicciones')
axis.legend()
axis.grid(alpha=0.2)
plt.show()

In [ ]:
errors = predictions.loc[~predictions['correct']].sort_values('confidence', ascending=False)
examples = errors.head(8) if not errors.empty else predictions.nsmallest(8, 'confidence')
title = 'Errores con mayor confianza' if not errors.empty else 'Predicciones correctas con menor confianza'

columns = 4
rows = max(1, int(np.ceil(len(examples) / columns)))
fig, axes = plt.subplots(rows, columns, figsize=(12, 3 * rows))
axes = np.atleast_1d(axes).ravel()
for axis, (_, row) in zip(axes, examples.iterrows()):
    image_path = Path(row['image_path'])
    if not image_path.is_absolute():
        image_path = Path(config['data']) / image_path
        if not image_path.exists():
            image_path = project_root / 'data' / 'synthetic_commands' / row['image_path']
    axis.imshow(Image.open(image_path).convert('RGB'))
    axis.set_title(f"real: {row['actual_class']}\npred.: {row['predicted_class']} ({row['confidence']:.2f})")
    axis.axis('off')
for axis in axes[len(examples):]:
    axis.axis('off')
fig.suptitle(title)
plt.tight_layout()
plt.show()

### Prueba en el ambiente independiente de Webots

Las capturas de `data/test_environment` no pertenecen a las particiones utilizadas durante el entrenamiento. Esta evaluación aproxima la transición de clasificación almacenada a inferencia dentro del simulador.

In [ ]:
import torch
from torch import nn
from torchvision import models, transforms

checkpoint = torch.load(run_dir / 'best_model.pth', map_location='cpu', weights_only=True)
model = models.mobilenet_v3_small(weights=None)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(checkpoint['classes']))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
preprocess = transforms.Compose([
    transforms.Resize((checkpoint['image_size'], checkpoint['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(checkpoint['normalization_mean'], checkpoint['normalization_std']),
])

test_environment = project_root / 'data' / 'test_environment'
snapshot_paths = sorted(test_environment.glob('cycle_*.png'))
if not snapshot_paths:
    print('No hay capturas. Ejecute: .\scripts\run_test_environment.ps1 -Cycles 8 -Batch')
else:
    rows = []
    with torch.inference_mode():
        for path in snapshot_paths:
            probabilities = torch.softmax(model(preprocess(Image.open(path).convert('RGB')).unsqueeze(0)), dim=1)[0]
            confidence, index = probabilities.max(dim=0)
            rows.append({
                'path': path,
                'real': path.stem.split('_')[-1],
                'predicha': checkpoint['classes'][index.item()],
                'confianza': confidence.item(),
            })
    environment_predictions = pd.DataFrame(rows)
    environment_predictions['correcta'] = environment_predictions['real'] == environment_predictions['predicha']
    display(environment_predictions.drop(columns='path'))

In [ ]:
if snapshot_paths:
    columns = 4
    rows_count = int(np.ceil(len(environment_predictions) / columns))
    fig, axes = plt.subplots(rows_count, columns, figsize=(12, 3 * rows_count))
    axes = np.atleast_1d(axes).ravel()
    for axis, (_, row) in zip(axes, environment_predictions.iterrows()):
        axis.imshow(Image.open(row['path']).convert('RGB'))
        axis.set_title(f"real: {row['real']}\npred.: {row['predicha']} ({row['confianza']:.2f})")
        axis.axis('off')
    for axis in axes[len(environment_predictions):]:
        axis.axis('off')
    plt.tight_layout()
    plt.show()

### Interpretación para robótica

1. Identificar la clase con menor *recall*. Una omisión de `stop` tiene mayor impacto operativo que una confusión entre avance y giro.
2. Comparar validación, prueba y ambiente independiente. Una caída progresiva indica menor capacidad de generalización.
3. Revisar oclusión, escala, iluminación y perspectiva en los errores con confianza alta.
4. Definir un umbral de confianza y una acción segura cuando ninguna predicción lo supere.
5. La validación con imágenes reales sigue siendo necesaria: el desempeño en simulación no elimina la brecha sintético-real.